<div class="alert alert-block" style="background-color: #51585eff; border-color: #42474aff; color: white;">
<center> <h1> Syntravel - Travel Diaries Generation </h1> </center> <br>
<center> <h2> Patterns Extraction </h2></center>

# Table of Contents

**1.[Importing Libraries and Loading Data](#importing-libraries-and-data)**  
   - [1.1 Importing Libraries](#importing-libraries)  
   - [1.2 Loading & Reading Data](#loading-and-reading-data)  

**2. [Rebuild subgroups from saved CSV](#rebuild)**

**3. [Run the CoT pipeline](#run)**
- [3.1 Quick Inspection](#quick)



# 1. Importing Libraries and Loading Data  <a class="anchor" id="importing-libraries-and-data"></a>

## 1.1. Importing Libraries <a class="anchor" id="importing-libraries"></a>

In [13]:
import os
if not os.path.isdir('llm_config'):
    os.chdir('..')

import sys
import json
import pandas as pd
import numpy as np
import shutil, datetime

## functions imports
from llm_config.llm_config import *
from Helpers.cot_patterns import *

## 1.2. Loading and Reading Data <a class="anchor" id="loading-and-reading-data"></a>

In [4]:
with open('Json_files/persona_objects.json', encoding='utf-8') as f:
    persona_objects = json.load(f)

print(f'Loaded {len(persona_objects)} persona objects')

Loaded 190 persona objects


# 2. Rebuild subgroups from saved CSV  <a class="anchor" id="rebuild"></a>

In [5]:
#odin_train.csv already has the group_key column, so we group by it and attach metadata from persona objects

odin_train = pd.read_csv(
    r'ODiN (DATA)/DATAVERSE/intermediarie_csvs/odin_train.csv',
    low_memory=False
)
print(f'Loaded odin_train: {odin_train.shape}')

Loaded odin_train: (101199, 125)


In [6]:
persona_meta = {p['group_key']: p for p in persona_objects}

subgroups = []
for group_key, gdf in odin_train.groupby('group_key', dropna=False):
    meta = persona_meta.get(group_key, {})
    subgroups.append({
        'group_key'  : group_key,
        'base_group' : meta.get('base_group', group_key),
        'split_dim'  : meta.get('split_dim'),
        'split_value': meta.get('income_level'),
        'df'         : gdf,
    })

print(f'Rebuilt {len(subgroups)} subgroups')

Rebuilt 190 subgroups


# 3. Run the CoT pipeline <a class="anchor" id="run"></a>

In [9]:
## Full run for all personas
OUTPUT_FILE = 'Json_files/cot_results.json'

results = run_cot_pipeline(
    persona_objects=persona_objects,
    subgroups=subgroups,
    output_file=OUTPUT_FILE,
    provider='groq_cot',  
    n_step1=8,
    n_step2=4,
)

print(f'Done. {len(results)} groups saved to {OUTPUT_FILE}')

[1/190] SKIP homemaker | 65+ | weekday
[2/190] SKIP employed | 51-66 | weekday | Income=Above median
[3/190] SKIP employed | 51-66 | weekday | Income=Below median
[4/190] SKIP employed | 51-66 | weekday | Income=High income
[5/190] SKIP employed | 51-66 | weekday | Income=Low income
[6/190] SKIP employed | 51-66 | weekday | Income=Median
[7/190] SKIP inactive | 51-66 | weekday | Income=Above median
[8/190] SKIP inactive | 51-66 | weekday | Income=Below median
[9/190] SKIP inactive | 51-66 | weekday | Income=High income
[10/190] SKIP inactive | 51-66 | weekday | Income=Low income
[11/190] SKIP inactive | 51-66 | weekday | Income=Median
[12/190] SKIP employed | 36-40 | weekday | Income=Above median
[13/190] SKIP employed | 36-40 | weekday | Income=Below median
[14/190] SKIP employed | 36-40 | weekday | Income=High income
[15/190] SKIP employed | 36-40 | weekday | Income=Low income
[16/190] SKIP employed | 36-40 | weekday | Income=Median
[17/190] SKIP retired | 65+ | weekend | Income=Abov


## 3.1 Quick inspection <a class="anchor" id="quick"></a>

In [19]:
sample_key = next(iter(results))
print(f'Group: {sample_key}\n')
print('--- initial_pattern ---')
print(results[sample_key]['initial_pattern'])
print('\n--- combined_output ---')
print(results[sample_key]['combined_output'])

Group: homemaker | 65+ | weekday

--- initial_pattern ---
This group of homemakers aged 65+ exhibits a distinct mobility pattern, characterized by a preference for short-distance trips, often on foot or by passenger car, with a focus on shopping and social visits. Their travel rhythm is centered around daytime hours, particularly between 9am and 12pm, with a notable emphasis on local trips within a 2.5 km radius. The group's mobility is likely influenced by their age, household composition, and urbanization level, which may limit their travel frequency and distance. Overall, this group's mobility pattern is marked by a strong emphasis on local, low-frequency travel, setting them apart from the broader Dutch population. Their mobility is also relatively simple, with fewer trips and a more predictable daily routine, reflecting their life stage and household circumstances.

--- combined_output ---
**1. TRAJECTORY VALIDATION**

Trajectory 1: Not consistent, suggests a subgroup as the dista

In [ ]:
# ONE-TIME CoT REPAIR — run 2026-07-03
# 7 persona groups were saved with empty initial_pattern/combined_output during
# the original CoT run, caused by a transient Groq rate-limit exhaustion in
# _call_with_retry (returns None after 5 failed attempts, which was cached as-is).
# Fix: delete the 7 keys so run_cot_pipeline's "skip if key present" guard no
# longer skips them, then re-run the pipeline to regenerate them (they succeeded
# on retry, confirming the failure was transient, not structural).
# On a clean cot_results.json these keys won't exist, so this cell is a no-op.
# Kept for reproducibility.
# Code starts here

# path = 'Json_files/cot_results.json'
# FAILED = [
#     'retired | 65+ | weekday | Income=Above median',
#     'retired | 65+ | weekday | Income=Below median',
#     'retired | 65+ | weekday | Income=High income',
#     'retired | 65+ | weekday | Income=Low income',
#     'retired | 65+ | weekday | Income=Median',
#     'incapacitated | 31-35 | weekday',
#     'employed | 31-35 | weekend | Income=Above median',
# ]

# cot = json.load(open(path, encoding='utf-8'))
# to_remove = [k for k in FAILED if k in cot]

# if to_remove:
#     stamp = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
#     shutil.copy(path, f'Json_files/cot_results.backup_{stamp}.json')
#     for k in to_remove:
#         cot.pop(k, None)
#     json.dump(cot, open(path, 'w', encoding='utf-8'), indent=2, ensure_ascii=False)
#     print(f'Repaired: removed {len(to_remove)} empty group(s); backup saved. '
#           f'Now {len(cot)} groups — re-run the pipeline cell to regenerate them.')
# else:
#     print(f'No repair needed: all target groups already present ({len(cot)} groups).')